# Model refinement - tahap 1: audit OOF

Tujuan: menemukan kesalahan yang dapat ditindaklanjuti sebelum mengubah
representasi fitur, tuning, atau ensemble. Notebook ini belum melatih model.

**Checkpoint tetap:** `ensemble_v08/20261005T121427960270Z`.
Dataset dan laporan dibaca melalui `src.config`; tidak memilih folder terbaru.
Notebook 1–4 dan laporan checkpoint tidak diubah.

Target positif adalah default (`1`). Penalti utama adalah `5 × FN + FP`,
bukan nominal kerugian uang. Threshold lama dipakai tanpa dioptimalkan ulang.
Seluruh diagnostik menggunakan training OOF; test tidak diinspeksi atau dievaluasi.
Skor ini tetap hasil pengembangan, karena parameter dan threshold sudah dipilih
menggunakan training. Audit tidak menghilangkan bias seleksi tersebut.




In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from hashlib import sha256
from importlib.metadata import version
from itertools import combinations
from pathlib import Path
import json
import sys
import zipfile

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value: object) -> None:
        """Tampilkan tabel saat audit dijalankan tanpa notebook kernel."""
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, log_loss, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold

# Mendukung notebook dibuka dari project root maupun folder notebooks.
for candidate_root in (Path.cwd(), Path.cwd().parent):
    if (candidate_root / "src" / "config.py").is_file():
        sys.path.insert(0, str(candidate_root))
        break

from src import config
from src.data_loader import load_and_split

CHECKPOINT_ID: str = "20261005T121427960270Z"
CHECKPOINT_DIR: Path = config.EXPERIMENTS_DIR / "ensemble_v08" / CHECKPOINT_ID
if not CHECKPOINT_DIR.is_dir():
    raise FileNotFoundError(f"Checkpoint tidak ditemukan: {CHECKPOINT_DIR}")

metadata = json.loads(
    (CHECKPOINT_DIR / config.EXPERIMENT_FILES["metadata"]).read_text(encoding="utf-8")
)
stored_summary = pd.read_csv(CHECKPOINT_DIR / config.EXPERIMENT_FILES["summary"],
                             float_precision="round_trip")

dataset_hash = sha256(config.DATA_PATH.read_bytes()).hexdigest()
if dataset_hash != metadata["dataset_sha256"]:
    raise ValueError("Hash dataset berbeda dari checkpoint.")
for key, expected in {
    "cv_folds": config.CV_FOLDS,
    "cv_random_state": config.CV_RANDOM_STATE,
    "false_negative_cost": config.FALSE_NEGATIVE_COST,
    "false_positive_cost": config.FALSE_POSITIVE_COST,
}.items():
    if metadata[key] != expected:
        raise ValueError(f"Konfigurasi {key} berbeda dari checkpoint.")
if metadata.get("test_evaluated") is not False:
    raise ValueError("Checkpoint tidak menyatakan test_evaluated=False.")

split = load_and_split()
X_train, y_train = split.X_train.copy(), split.y_train.copy()
del split  # Notebook hanya menyimpan partisi training.
if not X_train.index.is_unique or not X_train.index.equals(y_train.index):
    raise ValueError("Index training harus unik dan selaras.")
y = y_train.to_numpy(dtype=int)
expected_folds = np.zeros(len(y), dtype=int)
cv = StratifiedKFold(n_splits=config.CV_FOLDS, shuffle=True,
                    random_state=config.CV_RANDOM_STATE)
for fold, (_, positions) in enumerate(cv.split(X_train, y_train), start=1):
    expected_folds[positions] = fold


def score_at_threshold(probabilities: np.ndarray, threshold: float) -> dict[str, float | int]:
    """Hitung metrik pada label training dan threshold yang diberikan."""
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()
    cost = config.FALSE_NEGATIVE_COST * fn + config.FALSE_POSITIVE_COST * fp
    return {
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "total_cost": float(cost), "cost_per_1000": float(cost / len(y) * 1000),
        "f1": float(f1_score(y, predicted, zero_division=0)),
        "recall_pct": float(100 * tp / (tp + fn)),
        "precision_pct": float(100 * tp / (tp + fp)) if tp + fp else 0.0,
        "flagged_pct": float(100 * predicted.mean()),
        "roc_auc": float(roc_auc_score(y, probabilities)),
        "average_precision": float(average_precision_score(y, probabilities)),
        "brier_score": float(brier_score_loss(y, probabilities)),
        "log_loss": float(log_loss(y, probabilities, labels=[0, 1])),
    }


candidate_ids = list(metadata["candidate_definitions"])
if (stored_summary.duplicated(["experiment_id", "policy"]).any()
        or len(stored_summary) != 2 * len(candidate_ids)
        or set(stored_summary["experiment_id"]) != set(candidate_ids)):
    raise ValueError("Ringkasan checkpoint tidak lengkap atau mengandung duplikat.")

oof: dict[str, pd.DataFrame] = {}
scores: dict[str, np.ndarray] = {}
thresholds: dict[str, float] = {}
decisions: dict[str, np.ndarray] = {}
validation_rows = []
summary_rows = []
for name in candidate_ids:
    table = pd.read_csv(
        CHECKPOINT_DIR / name / config.EXPERIMENT_FILES["oof_predictions"],
        float_precision="round_trip",
    )
    if (len(table) != len(y) or not table["source_row_index"].is_unique
            or not np.array_equal(table["source_row_index"], X_train.index)
            or not np.array_equal(table["y_true"], y)
            or not np.array_equal(table["fold"], expected_folds)):
        raise ValueError(f"Baris, label, atau fold {name} tidak cocok.")
    p = table["probability_default"].to_numpy(dtype=float)
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError(f"Probabilitas {name} tidak valid.")
    rows = stored_summary.loc[stored_summary["experiment_id"].eq(name)]
    if set(rows["policy"]) != {"cost", "f1_diagnostic"}:
        raise ValueError(f"Kebijakan {name} tidak lengkap.")
    for _, stored in rows.iterrows():
        threshold = float(stored["threshold"])
        if not np.isfinite(threshold) or not 0 <= threshold <= 1:
            raise ValueError(f"Threshold {name} tidak valid.")
        measured = score_at_threshold(p, threshold)
        for metric, value in measured.items():
            if metric in stored.index and not np.isclose(
                value, float(stored[metric]), rtol=1e-10, atol=1e-10
            ):
                raise ValueError(f"Metrik {name}/{stored['policy']}/{metric} berbeda.")
        if stored["policy"] == "cost":
            thresholds[name] = threshold
            decisions[name] = (p >= threshold).astype(int)
            if not np.array_equal(table["prediction"], decisions[name]):
                raise ValueError(f"Keputusan tersimpan {name} berbeda.")
            summary_rows.append({"candidate": name, "threshold": threshold, **measured})
    oof[name], scores[name] = table, p
    validation_rows.append({"candidate": name, "train_rows": len(table),
                            "rows_labels_folds_metrics": "PASS"})

for name, definition in metadata["candidate_definitions"].items():
    if "weights" in definition:
        members = [scores[f"single_{member}_v08"] for member in definition["members"]]
        expected = np.average(np.column_stack(members), axis=1,
                              weights=definition["weights"])
        if not np.allclose(scores[name], expected, rtol=0, atol=1e-14):
            raise ValueError(f"Rumus blending {name} berbeda.")

validation = pd.DataFrame(validation_rows)
candidate_summary = pd.DataFrame(summary_rows).sort_values("cost_per_1000").reset_index(drop=True)
PRIMARY_CANDIDATE: str = "blend_two_v08"  # Kandidat utama dari checkpoint, bukan dipilih ulang.
SINGLE_MODELS: tuple[str, ...] = ("single_xgb_v08", "single_lgbm_v08", "single_cat_v08")
package_versions = {package: version(package) for package in ("numpy", "pandas", "scikit-learn")}
version_comparison = pd.DataFrame([
    {"package": package, "checkpoint": metadata["package_versions"][package], "current": current}
    for package, current in package_versions.items()
])
print("VALIDASI LULUS — hanya train OOF; tidak ada training atau evaluasi test.")
display(validation)
display(candidate_summary[["candidate", "threshold", "cost_per_1000", "roc_auc",
                           "average_precision", "f1", "recall_pct", "precision_pct",
                           "flagged_pct", "brier_score", "log_loss"]].round(4))
display(version_comparison)


VALIDASI LULUS — hanya train OOF; tidak ada training atau evaluasi test.


,candidate,train_rows,rows_labels_folds_metrics
0,single_xgb_v08,24000,PASS
1,single_lgbm_v08,24000,PASS
2,single_cat_v08,24000,PASS
3,blend_two_v08,24000,PASS
4,blend_three_v08,24000,PASS
5,stack_two_lr_v08,24000,PASS


,candidate,threshold,cost_per_1000,roc_auc,average_precision,f1,recall_pct,precision_pct,flagged_pct,brier_score,log_loss
0,blend_two_v08,0.1638,534.7083,0.7878,0.5596,0.5142,75.0047,39.1159,42.4167,0.1332,0.4246
1,blend_three_v08,0.1666,535.2917,0.7881,0.5623,0.5168,74.1759,39.6576,41.3750,0.1329,0.4242
2,single_xgb_v08,0.1620,535.6250,0.7864,0.5562,0.5115,75.5321,38.6692,43.2083,0.1335,0.4260
3,stack_two_lr_v08,0.1435,536.1250,0.7877,0.5601,0.5093,76.0407,38.2872,43.9333,0.1339,0.4281
4,single_lgbm_v08,0.1776,538.8750,0.7875,0.5606,0.5227,71.9533,41.0400,38.7833,0.1332,0.4246
5,single_cat_v08,0.1437,539.3333,0.7850,0.5571,0.4965,79.2051,36.1596,48.4542,0.1337,0.4270


,package,checkpoint,current
0,numpy,2.4.6,2.4.6
1,pandas,3.0.6,3.0.6
2,scikit-learn,1.9.1,1.9.1


In [2]:
predicted = decisions[PRIMARY_CANDIDATE]
amount_columns = [*config.BILL_COLUMNS, *config.PAYMENT_COLUMNS]
repayment = X_train.loc[:, list(config.REPAYMENT_STATUS_COLUMNS)]
if not np.isfinite(X_train.to_numpy(dtype=float)).all():
    raise ValueError("Audit fitur membutuhkan nilai finite, tanpa missing.")
segment_definitions = {
    "PAY_0_code": X_train["PAY_0"].map(lambda value: f"code {int(value)}"),
    "all_amounts_zero": X_train[amount_columns].eq(0).all(axis=1).map({True: "yes", False: "no"}),
    "any_positive_delay_code": repayment.gt(0).any(axis=1).map({True: "yes", False: "no"}),
}


def summarize_segments(segment_name: str, groups: pd.Series) -> pd.DataFrame:
    """Ringkas kesalahan kandidat utama dalam kelompok fitur training."""
    if not groups.index.equals(X_train.index) or groups.isna().any():
        raise ValueError(f"Kelompok {segment_name} tidak selaras atau missing.")
    work = pd.DataFrame({"group": groups.to_numpy(), "y": y, "prediction": predicted})
    rows = []
    total_fn = int(((y == 1) & (predicted == 0)).sum())
    total_fp = int(((y == 0) & (predicted == 1)).sum())
    for group, part in work.groupby("group", sort=True):
        tn, fp, fn, tp = confusion_matrix(part["y"], part["prediction"], labels=[0, 1]).ravel()
        default_count, non_default_count = int(tp + fn), int(tn + fp)
        rows.append({
            "segment": segment_name, "group": group, "rows": len(part),
            "defaults": default_count, "non_defaults": non_default_count,
            "default_rate_pct": 100 * default_count / len(part),
            "fn": int(fn), "fp": int(fp),
            "fn_rate_pct": 100 * fn / default_count if default_count else np.nan,
            "fp_rate_pct": 100 * fp / non_default_count if non_default_count else np.nan,
            "share_all_fn_pct": 100 * fn / total_fn if total_fn else 0.0,
            "share_all_fp_pct": 100 * fp / total_fp if total_fp else 0.0,
            "cost_per_1000": 1000 * (config.FALSE_NEGATIVE_COST * fn
                                       + config.FALSE_POSITIVE_COST * fp) / len(part),
        })
    return pd.DataFrame(rows)


segment_errors = pd.concat(
    [summarize_segments(name, groups) for name, groups in segment_definitions.items()],
    ignore_index=True,
)
for name in segment_definitions:
    print(f"SEGMENT: {name}")
    display(segment_errors.loc[segment_errors["segment"].eq(name)].round(2))

distance = np.abs(scores[PRIMARY_CANDIDATE] - thresholds[PRIMARY_CANDIDATE])
error_distance = pd.DataFrame([
    {"error": label, "rows": int(mask.sum()),
     "within_0.02_of_threshold_pct": float(100 * (distance[mask] <= 0.02).mean())}
    for label, mask in (
        ("FN", (y == 1) & (predicted == 0)),
        ("FP", (y == 0) & (predicted == 1)),
    )
])
print("KESALAHAN DEKAT THRESHOLD (jarak probabilitas <= 0.02)")
display(error_distance.round(2))


SEGMENT: PAY_0_code


,segment,group,rows,defaults,non_defaults,default_rate_pct,fn,fp,fn_rate_pct,fp_rate_pct,share_all_fn_pct,share_all_fp_pct,cost_per_1000
0,PAY_0_code,code -1,4527,753,3774,16.63,268,1382,35.59,36.62,20.20,22.30,601.28
1,PAY_0_code,code -2,2198,279,1919,12.69,153,385,54.84,20.06,11.53,6.21,523.20
2,PAY_0_code,code 0,11781,1508,10273,12.80,857,2069,56.83,20.14,64.58,33.38,539.34
3,PAY_0_code,code 1,3000,1031,1969,34.37,49,1606,4.75,81.56,3.69,25.91,617.00
4,PAY_0_code,code 2,2138,1483,655,69.36,0,655,0.00,100.00,0.00,10.57,306.36
5,PAY_0_code,code 3,245,188,57,76.73,0,57,0.00,100.00,0.00,0.92,232.65
6,PAY_0_code,code 4,58,38,20,65.52,0,20,0.00,100.00,0.00,0.32,344.83
7,PAY_0_code,code 5,18,8,10,44.44,0,10,0.00,100.00,0.00,0.16,555.56
8,PAY_0_code,code 6,10,6,4,60.00,0,4,0.00,100.00,0.00,0.06,400.00
9,PAY_0_code,code 7,9,7,2,77.78,0,2,0.00,100.00,0.00,0.03,222.22


SEGMENT: all_amounts_zero


,segment,group,rows,defaults,non_defaults,default_rate_pct,fn,fp,fn_rate_pct,fp_rate_pct,share_all_fn_pct,share_all_fp_pct,cost_per_1000
11,all_amounts_zero,no,23358,5069,18289,21.70,1327,5801,26.18,31.72,100.0,93.59,532.41
12,all_amounts_zero,yes,642,240,402,37.38,0,397,0.00,98.76,0.0,6.41,618.38


SEGMENT: any_positive_delay_code


,segment,group,rows,defaults,non_defaults,default_rate_pct,fn,fp,fn_rate_pct,fp_rate_pct,share_all_fn_pct,share_all_fp_pct,cost_per_1000
13,any_positive_delay_code,no,15896,1830,14066,11.51,1201,2366,65.63,16.82,90.5,38.17,526.61
14,any_positive_delay_code,yes,8104,3479,4625,42.93,126,3832,3.62,82.85,9.5,61.83,550.59


KESALAHAN DEKAT THRESHOLD (jarak probabilitas <= 0.02)


,error,rows,within_0.02_of_threshold_pct
0,FN,1327,18.31
1,FP,6198,16.91


In [3]:
REVIEW_FRACTION: float = 0.30
quota = int(np.ceil(REVIEW_FRACTION * len(y)))
quota_decisions: dict[str, np.ndarray] = {}
workload_rows = []
for name in SINGLE_MODELS:
    ranking = np.lexsort((X_train.index.to_numpy(), -scores[name]))
    selected = np.zeros(len(y), dtype=int)
    selected[ranking[:quota]] = 1
    quota_decisions[name] = selected
    tn, fp, fn, tp = confusion_matrix(y, selected, labels=[0, 1]).ravel()
    workload_rows.append({
        "candidate": name, "flagged_rows": int(selected.sum()),
        "recall_pct": float(100 * tp / (tp + fn)),
        "precision_pct": float(100 * tp / (tp + fp)),
        "fn": int(fn), "fp": int(fp),
        "cost_per_1000": float(1000 * (config.FALSE_NEGATIVE_COST * fn
                                      + config.FALSE_POSITIVE_COST * fp) / len(y)),
    })
equal_workload = pd.DataFrame(workload_rows)


def compare_pair(name_a: str, name_b: str, policy: str,
                 decision_a: np.ndarray, decision_b: np.ndarray) -> dict[str, str | int | float]:
    """Hitung kesalahan bersama dan perubahan penalti dari A ke B."""
    positive, negative = y == 1, y == 0
    rescued_fn = int((positive & (decision_a == 0) & (decision_b == 1)).sum())
    added_fn = int((positive & (decision_a == 1) & (decision_b == 0)).sum())
    removed_fp = int((negative & (decision_a == 1) & (decision_b == 0)).sum())
    added_fp = int((negative & (decision_a == 0) & (decision_b == 1)).sum())
    return {
        "policy": policy, "model_a": name_a, "model_b": name_b,
        "disagreement_pct": float(100 * (decision_a != decision_b).mean()),
        "joint_fn": int((positive & (decision_a == 0) & (decision_b == 0)).sum()),
        "joint_fp": int((negative & (decision_a == 1) & (decision_b == 1)).sum()),
        "fn_rescued_by_b": rescued_fn, "fn_added_by_b": added_fn,
        "fp_removed_by_b": removed_fp, "fp_added_by_b": added_fp,
        "net_cost_reduction_b_vs_a": float(
            config.FALSE_NEGATIVE_COST * (rescued_fn - added_fn)
            + config.FALSE_POSITIVE_COST * (removed_fp - added_fp)
        ),
    }


pair_rows = []
for policy, decision_set in (("individual_cost_threshold", decisions),
                              ("equal_workload_30pct", quota_decisions)):
    for name_a, name_b in combinations(SINGLE_MODELS, 2):
        pair_rows.append(compare_pair(name_a, name_b, policy,
                                      decision_set[name_a], decision_set[name_b]))
model_complementarity = pd.DataFrame(pair_rows)
for policy in model_complementarity["policy"].unique():
    print(f"PAIR DIAGNOSTICS: {policy}")
    display(model_complementarity.loc[model_complementarity["policy"].eq(policy)].round(2))
print("BEBAN PEMERIKSAAN SAMA: 30% TRAIN")
display(equal_workload.round(4))


PAIR DIAGNOSTICS: individual_cost_threshold


,policy,model_a,model_b,disagreement_pct,joint_fn,joint_fp,fn_rescued_by_b,fn_added_by_b,fp_removed_by_b,fp_added_by_b,net_cost_reduction_b_vs_a
0,individual_cost_threshold,single_xgb_v08,single_lgbm_v08,6.61,1258,5267,41,231,1093,221,-78.0
1,individual_cost_threshold,single_xgb_v08,single_cat_v08,9.30,1027,5950,272,77,410,1474,-89.0
2,individual_cost_threshold,single_lgbm_v08,single_cat_v08,11.26,1074,5327,415,30,161,2097,-11.0


PAIR DIAGNOSTICS: equal_workload_30pct


,policy,model_a,model_b,disagreement_pct,joint_fn,joint_fp,fn_rescued_by_b,fn_added_by_b,fp_removed_by_b,fp_added_by_b,net_cost_reduction_b_vs_a
3,equal_workload_30pct,single_xgb_v08,single_lgbm_v08,3.59,1816,3487,106,105,326,325,6.0
4,equal_workload_30pct,single_xgb_v08,single_cat_v08,4.99,1781,3353,141,139,460,458,12.0
5,equal_workload_30pct,single_lgbm_v08,single_cat_v08,4.60,1795,3385,126,125,427,426,6.0


BEBAN PEMERIKSAAN SAMA: 30% TRAIN


,candidate,flagged_rows,recall_pct,precision_pct,fn,fp,cost_per_1000
0,single_xgb_v08,7200,63.7973,47.0417,1922,3813,559.2917
1,single_lgbm_v08,7200,63.8162,47.0556,1921,3812,559.0417
2,single_cat_v08,7200,63.8350,47.0694,1920,3811,558.7917


In [4]:
THRESHOLD_OFFSETS: tuple[float, ...] = (-0.03, -0.02, -0.01, 0.0, 0.01, 0.02, 0.03)
sensitivity_rows = []
for name in (PRIMARY_CANDIDATE, "single_xgb_v08"):
    base_threshold = thresholds[name]
    base_cost = score_at_threshold(scores[name], base_threshold)["cost_per_1000"]
    for offset in THRESHOLD_OFFSETS:
        threshold = float(np.clip(base_threshold + offset, 0, 1))
        measured = score_at_threshold(scores[name], threshold)
        sensitivity_rows.append({
            "candidate": name, "offset": offset, "threshold": threshold,
            **{key: measured[key] for key in (
                "cost_per_1000", "fn", "fp", "recall_pct", "precision_pct", "flagged_pct"
            )},
            "delta_cost_per_1000": float(measured["cost_per_1000"] - base_cost),
        })
threshold_sensitivity = pd.DataFrame(sensitivity_rows)
display(threshold_sensitivity.round(4))


,candidate,offset,threshold,cost_per_1000,fn,fp,recall_pct,precision_pct,flagged_pct,delta_cost_per_1000
0,blend_two_v08,-0.03,0.1338,544.7917,909,8530,82.8781,34.0294,53.8750,10.0833
1,blend_two_v08,-0.02,0.1438,547.2083,1084,7713,79.5818,35.3912,49.7417,12.5000
2,blend_two_v08,-0.01,0.1538,540.2917,1213,6902,77.1520,37.2431,45.8250,5.5833
3,blend_two_v08,0.00,0.1638,534.7083,1327,6198,75.0047,39.1159,42.4167,0.0000
4,blend_two_v08,0.01,0.1738,539.0417,1454,5667,72.6125,40.4852,39.6750,4.3333
5,blend_two_v08,0.02,0.1838,540.4167,1564,5150,70.5406,42.1023,37.0625,5.7083
6,blend_two_v08,0.03,0.1938,543.6667,1660,4748,68.7323,43.4560,34.9875,8.9583
7,single_xgb_v08,-0.03,0.1320,549.6667,885,8767,83.3302,33.5380,54.9625,14.0417
8,single_xgb_v08,-0.02,0.1420,541.9583,1035,7832,80.5048,35.3048,50.4417,6.3333
9,single_xgb_v08,-0.01,0.1520,539.5417,1175,7074,77.8678,36.8844,46.7000,3.9167


In [5]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
AUDIT_DIR: Path = config.REPORTS_DIR / "audits" / "ensemble_v08" / run_id
AUDIT_DIR.mkdir(parents=True, exist_ok=False)
audit_tables = {
    "validation.csv": validation,
    "candidate_summary.csv": candidate_summary,
    "segment_errors.csv": segment_errors,
    "error_distance.csv": error_distance,
    "model_complementarity.csv": model_complementarity,
    "equal_workload.csv": equal_workload,
    "threshold_sensitivity.csv": threshold_sensitivity,
}
for filename, table in audit_tables.items():
    table.to_csv(AUDIT_DIR / filename, index=False)

source_hashes = {
    "checkpoint_metadata": sha256((CHECKPOINT_DIR / config.EXPERIMENT_FILES["metadata"]).read_bytes()).hexdigest(),
    "checkpoint_summary": sha256((CHECKPOINT_DIR / config.EXPERIMENT_FILES["summary"]).read_bytes()).hexdigest(),
    **{name: sha256((CHECKPOINT_DIR / name / config.EXPERIMENT_FILES["oof_predictions"]).read_bytes()).hexdigest()
       for name in candidate_ids},
}
audit_metadata = {
    "run_id": run_id, "checkpoint": CHECKPOINT_DIR.relative_to(config.PROJECT_ROOT).as_posix(),
    "dataset_sha256": dataset_hash, "training_rows": len(y),
    "training_index_sha256": sha256(X_train.index.to_numpy(dtype="<i8").tobytes()).hexdigest(),
    "scope": "training_development_oof_diagnostics", "test_evaluated": False,
    "model_training_performed": False, "primary_candidate": PRIMARY_CANDIDATE,
    "false_negative_cost": config.FALSE_NEGATIVE_COST,
    "false_positive_cost": config.FALSE_POSITIVE_COST,
    "review_fraction_diagnostic": REVIEW_FRACTION, "threshold_offsets": THRESHOLD_OFFSETS,
    "package_versions": package_versions, "source_hashes": source_hashes,
    "audit_module_hashes": {
        name: sha256((config.PROJECT_ROOT / "src" / f"{name}.py").read_bytes()).hexdigest()
        for name in ("config", "data_loader")
    },
}
(AUDIT_DIR / "metadata.json").write_text(
    json.dumps(audit_metadata, indent=2, allow_nan=False), encoding="utf-8"
)
ZIP_PATH: Path = AUDIT_DIR / "audit_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in [*audit_tables, "metadata.json"]:
        archive.write(AUDIT_DIR / filename, arcname=filename)
print(f"KIRIM FILE INI: {ZIP_PATH}")
print(f"Ukuran: {ZIP_PATH.stat().st_size / 1024:.1f} KB")



KIRIM FILE INI: D:\proj-1\reports\audits\ensemble_v08\20261009T051826304022Z\audit_results.zip
Ukuran: 5.0 KB


## Kesimpulan audit OOF ensemble

Seluruh kandidat lolos pemeriksaan kesesuaian dataset, anggota training,
label, fold, metrik, dan perhitungan blending.

Temuan utama:

- Blend XGBoost–LightGBM memiliki penalti terendah: 534,7083 per 1.000
  observasi. Keunggulannya terhadap XGBoost tunggal hanya 22 unit penalti
  pada 24.000 observasi, sehingga belum membuktikan keuntungan yang stabil.
- Sebanyak 1.201 dari 1.327 FN (90,50%) berasal dari nasabah tanpa kode
  keterlambatan positif selama enam bulan.
- Pada kuota pemeriksaan yang sama, kemampuan ketiga model dasar hampir
  setara, meskipun sebagian nasabah yang ditandai berbeda.
- Pergeseran threshold menukar FN dengan FP. Hanya 18,31% FN berada dalam
  jarak probabilitas 0,02 dari threshold blend.
- Kelompok dengan seluruh nominal nol memiliki default rate 37,38%.
  Kelompok ini dipertahankan; belum ada dasar untuk menghapusnya.

Keputusan:
Prioritaskan pemeriksaan pola keuangan pada kelompok tanpa kode
keterlambatan positif sebelum menambah kompleksitas ensemble.

Seluruh hasil merupakan diagnostik development OOF. Test set belum
dievaluasi, dan penalti merupakan asumsi relatif, bukan kerugian uang aktual.

## Pemeriksaan pola keuangan pada kelompok tanpa keterlambatan positif

Tujuan:
Mencari perbedaan pola keuangan yang dapat dijadikan hipotesis untuk
eksperimen fitur berikutnya.

Kelompok hasil prediksi:

- TN: non-default yang tidak ditandai.
- FP: non-default yang ditandai.
- FN: default yang tidak ditandai.
- TP: default yang ditandai.

Perbandingan utama adalah FN versus TN: keduanya tidak ditandai oleh
model, tetapi memiliki hasil aktual berbeda. FN versus TP membantu
memahami karakteristik default yang terlewat dan yang tertangkap.

Ringkasan menggunakan median dan kuartil karena nominal keuangan
memiliki distribusi yang tidak simetris. Kuartil membantu menunjukkan
tumpang tindih yang tidak terlihat dari median saja.

Fitur yang diperiksa meliputi nominal, variasi tagihan, pembayaran nol,
tagihan terbaru terhadap limit, serta perubahan nominal yang dinormalisasi
dengan limit. Perubahan dihitung sebagai rata-rata tiga bulan terbaru
dikurangi rata-rata tiga bulan terlama.

Tidak ada fitting, perubahan threshold, atau pemilihan aturan keputusan
dalam pemeriksaan ini.

In [6]:
def build_financial_view(X: pd.DataFrame) -> pd.DataFrame:
    """Hitung ringkasan keuangan per baris untuk diagnosis, tanpa fitting."""
    required = [
        "LIMIT_BAL",
        "AGE",
        *config.BILL_COLUMNS,
        *config.PAYMENT_COLUMNS,
    ]

    if not set(required).issubset(X.columns):
        raise ValueError("Kolom keuangan tidak lengkap.")

    if not np.isfinite(X[required].to_numpy(dtype=float)).all():
        raise ValueError("Fitur keuangan mengandung missing atau infinity.")

    if X["LIMIT_BAL"].le(0).any():
        raise ValueError("Rasio terhadap limit memerlukan LIMIT_BAL > 0.")

    bills = X.loc[:, list(config.BILL_COLUMNS)]
    payments = X.loc[:, list(config.PAYMENT_COLUMNS)]
    limit = X["LIMIT_BAL"]

    # Urutan kolom: bulan terbaru menuju bulan terlama.
    bill_change = (
        bills.iloc[:, :3].mean(axis=1)
        - bills.iloc[:, 3:].mean(axis=1)
    )
    payment_change = (
        payments.iloc[:, :3].mean(axis=1)
        - payments.iloc[:, 3:].mean(axis=1)
    )

    return pd.DataFrame(
        {
            "LIMIT_BAL": limit,
            "AGE": X["AGE"],
            "bill_mean": bills.mean(axis=1),
            "bill_std": bills.std(axis=1, ddof=0),
            "payment_mean": payments.mean(axis=1),
            "zero_payment_months": payments.eq(0).sum(axis=1),
            "bill_latest_to_limit": bills.iloc[:, 0] / limit,
            "bill_change_to_limit": bill_change / limit,
            "payment_change_to_limit": payment_change / limit,
        },
        index=X.index,
    )


if (
    not X_train.index.equals(y_train.index)
    or not X_train.index.is_unique
):
    raise ValueError("Index fitur dan target harus unik dan selaras.")

audit_probabilities = scores[PRIMARY_CANDIDATE]

if (
    audit_probabilities.shape != (len(X_train),)
    or not np.isfinite(audit_probabilities).all()
    or ((audit_probabilities < 0) | (audit_probabilities > 1)).any()
):
    raise ValueError("Probabilitas audit tidak valid.")

audit_prediction = pd.Series(
    (
        audit_probabilities
        >= thresholds[PRIMARY_CANDIDATE]
    ).astype(int),
    index=X_train.index,
)

no_positive_delay = (
    ~X_train[list(config.REPAYMENT_STATUS_COLUMNS)]
    .gt(0)
    .any(axis=1)
)

financial_view = build_financial_view(X_train)

outcome = pd.Series(
    np.select(
        [
            y_train.eq(1) & audit_prediction.eq(0),
            y_train.eq(1) & audit_prediction.eq(1),
            y_train.eq(0) & audit_prediction.eq(1),
        ],
        ["FN", "TP", "FP"],
        default="TN",
    ),
    index=X_train.index,
    name="outcome",
)

group_order = ["TN", "FP", "FN", "TP"]
subset = financial_view.loc[no_positive_delay]
subset_outcome = outcome.loc[no_positive_delay]

financial_counts = (
    subset_outcome.value_counts()
    .reindex(group_order, fill_value=0)
    .rename_axis("outcome")
    .reset_index(name="rows")
)

financial_counts["share_subset_pct"] = (
    100 * financial_counts["rows"] / len(subset)
)

financial_medians = (
    subset.groupby(subset_outcome, observed=True)
    .median()
    .reindex(group_order)
    .T
    .rename_axis("feature")
)

# FN vs TN: keduanya tidak ditandai, tetapi hanya FN yang gagal bayar.
quartile_tables = {}

for group in ("FN", "TN"):
    group_data = subset.loc[subset_outcome.eq(group)]

    quartile_tables[group] = (
        group_data.quantile([0.25, 0.50, 0.75])
        .T
        .rename(
            columns={
                0.25: "q25",
                0.50: "median",
                0.75: "q75",
            }
        )
    )

financial_quartiles = (
    pd.concat(quartile_tables, axis=1)
    .rename_axis("feature")
)

print(
    f"Kandidat: {PRIMARY_CANDIDATE}; "
    f"threshold tetap: {thresholds[PRIMARY_CANDIDATE]:.6f}"
)
print("Kelompok: tidak ada kode repayment positif selama enam bulan.")

print("\nTABEL 1 — Jumlah kasus")
display(financial_counts.round(2))

print("\nTABEL 2 — Median fitur per kelompok hasil prediksi")
display(financial_medians.round(4))

print("\nTABEL 3 — Kuartil FN dan TN")
display(financial_quartiles.round(4))

Kandidat: blend_two_v08; threshold tetap: 0.163819
Kelompok: tidak ada kode repayment positif selama enam bulan.

TABEL 1 — Jumlah kasus


,outcome,rows,share_subset_pct
0,TN,11700,73.60
1,FP,2366,14.88
2,FN,1201,7.56
3,TP,629,3.96



TABEL 2 — Median fitur per kelompok hasil prediksi


outcome,TN,FP,FN,TP
feature,,,,
LIMIT_BAL,180000.0000,75000.0000,150000.0000,80000.0000
AGE,34.0000,36.0000,34.0000,36.0000
bill_mean,29919.9167,3199.0000,27339.1667,1876.0000
bill_std,7602.0714,1330.0769,7749.9051,1032.2873
payment_mean,4000.0000,937.1667,3176.6667,711.0000
zero_payment_months,0.0000,1.0000,0.0000,1.0000
bill_latest_to_limit,0.2920,0.0628,0.4974,0.0359
bill_change_to_limit,0.0204,0.0021,0.0299,0.0000
payment_change_to_limit,0.0044,0.0017,0.0047,0.0008



TABEL 3 — Kuartil FN dan TN


FN                                    TN  \
                                q25       median          q75         q25   
feature                                                                     
LIMIT_BAL                70000.0000  150000.0000  240000.0000  90000.0000   
AGE                         28.0000      34.0000      41.0000     29.0000   
bill_mean                 6241.1667   27339.1667   67413.0000   9315.7917   
bill_std                  2453.5725    7749.9051   17916.7701   2837.8547   
payment_mean              1580.1667    3176.6667    6667.3333   1951.3750   
zero_payment_months          0.0000       0.0000       1.0000      0.0000   
bill_latest_to_limit         0.0278       0.4974       0.9416      0.0385   
bill_change_to_limit        -0.0067       0.0299       0.2448     -0.0105   
payment_change_to_limit     -0.0029       0.0047       0.0184     -0.0022   

                                                   
                              median          q75  
feature                                            
LIMIT_BAL                180000.0000  280000.0000  
AGE                          34.0000      41.0000  
bill_mean                 29919.9167   74149.6667  
bill_std                   7602.0714   17324.9208  
payment_mean               4000.0000    8019.0000  
zero_payment_months           0.0000       0.0000  
bill_latest_to_limit          0.2920       0.7896  
bill_change_to_limit          0.0204       0.1478  
payment_change_to_limit       0.0044       0.0166

### Cara membaca hasil dan batas kesimpulan

Perbedaan median menjadi petunjuk untuk memilih fitur yang akan diuji,
bukan bukti bahwa fitur tersebut meningkatkan kemampuan prediksi.

Perhatikan FN versus TN:

- Apakah nominal pembayaran berbeda?
- Apakah tagihan relatif terhadap limit berbeda?
- Apakah frekuensi pembayaran nol atau perubahan nominal berbeda?
- Seberapa besar rentang kuartil kedua kelompok bertumpang tindih?

Kelompok ini ditentukan berdasarkan hasil model, sehingga temuan bersifat
deskriptif dan tidak menunjukkan hubungan sebab-akibat. Nama rasio
tagihan terhadap limit juga tidak diartikan sebagai persentase kewajiban
yang telah dilunasi.

Hipotesis yang menjanjikan akan diuji melalui ablation pada seluruh
training set, dengan fold yang sama dan parameter model tetap.
Nasabah tidak dihapus dan model khusus per kelompok belum dibuat.

### Kesimpulan analisis kesalahan dan hipotesis fitur

Pada kelompok tanpa kode repayment positif selama enam bulan,
model melewatkan 1.201 dari 1.830 kasus default.

Dibandingkan true negative, false negative memiliki median rasio
tagihan terbaru terhadap limit lebih tinggi (0,4974 vs 0,2920)
dan rata-rata pembayaran lebih rendah (3.176,67 vs 4.000).

Namun, distribusi kedua kelompok masih banyak bertumpang tindih.
Perbedaan ini merupakan temuan deskriptif, bukan bukti hubungan
kausal atau jaminan peningkatan performa.

Eksperimen berikut menguji dua paket fitur:

1. Agregat + rasio tagihan, pembayaran, dan perubahan tagihan.
2. Paket pertama + jumlah bulan tanpa pembayaran dan indikator
   seluruh nominal tagihan serta pembayaran bernilai nol.

Setiap paket diuji pada XGBoost dan LightGBM dengan parameter
model tetap dari eksperimen sebelumnya. Fitur asli dan encoding
kategorikal tetap dipertahankan.

Evaluasi menggunakan lima fold OOF pada development set.
Threshold setiap kandidat dipilih ulang dari prediksi OOF dengan
meminimalkan 5 × FN + FP. Threshold kandidat lama tidak diterapkan
pada model baru karena distribusi probabilitasnya dapat berubah.

Ukuran utama adalah biaya per 1.000 observasi. Average Precision,
ROC-AUC, F1, jumlah FP/FN, dan proporsi prediksi positif menjadi
diagnostik tambahan. Test set tetap dikarantina.

Hasil OOF digunakan untuk pemilihan kandidat dan dapat optimistis
karena development set telah dipakai dalam beberapa eksperimen.
Evaluasi akhir dilakukan setelah seluruh keputusan dikunci.

In [7]:
import hashlib
import importlib
import json
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from src import config
from src import feature_engineering, preprocess, experiments

importlib.reload(config)
importlib.reload(feature_engineering)
importlib.reload(preprocess)
importlib.reload(experiments)

CreditFeatureEngineer = feature_engineering.CreditFeatureEngineer
build_preprocessor = preprocess.build_preprocessor
run_oof_experiment = experiments.run_oof_experiment
save_experiment = experiments.save_experiment

with (CHECKPOINT_DIR / "metadata.json").open(
    encoding="utf-8"
) as file:
    reference_metadata = json.load(file)

dataset_hash = hashlib.sha256(
    config.DATA_PATH.read_bytes()
).hexdigest()

assert dataset_hash == reference_metadata["dataset_sha256"]
assert config.CV_FOLDS == reference_metadata["cv_folds"]
assert config.CV_RANDOM_STATE == reference_metadata["cv_random_state"]
assert config.FALSE_NEGATIVE_COST == reference_metadata["false_negative_cost"]
assert config.FALSE_POSITIVE_COST == reference_metadata["false_positive_cost"]

version_check = pd.DataFrame([
    {
        "package": package,
        "checkpoint": reference_metadata["package_versions"][package],
        "current": version(package),
    }
    for package in (
        "numpy", "pandas", "scikit-learn", "xgboost", "lightgbm"
    )
])
display(version_check)

if not version_check["checkpoint"].eq(version_check["current"]).all():
    raise RuntimeError(
        "Versi berbeda dari checkpoint. Kirim tabel versi sebelum lanjut."
    )

feature_packs = {
    "ratios": (
        *config.FE_AGGREGATE_COLUMNS,
        *config.FE_REFINEMENT_RATIO_COLUMNS,
    ),
    "ratios_behavior": (
        *config.FE_AGGREGATE_COLUMNS,
        *config.FE_REFINEMENT_RATIO_COLUMNS,
        *config.FE_REFINEMENT_BEHAVIOR_COLUMNS,
    ),
}

reference_oof = {}
for candidate in (
    "single_xgb_v08", "single_lgbm_v08", "blend_two_v08"
):
    table = pd.read_csv(
        CHECKPOINT_DIR / candidate / "oof_predictions.csv"
    )
    np.testing.assert_array_equal(
        table["source_row_index"].to_numpy(), X_train.index.to_numpy()
    )
    np.testing.assert_array_equal(
        table["y_true"].to_numpy(), y_train.to_numpy()
    )
    reference_oof[candidate] = table

display(pd.DataFrame([
    {"feature_pack": name, "derived_features": len(features)}
    for name, features in feature_packs.items()
]))

print("Pembanding selaras. Empat eksperimen siap dijalankan.")

,package,checkpoint,current
0,numpy,2.4.6,2.4.6
1,pandas,3.0.6,3.0.6
2,scikit-learn,1.9.1,1.9.1
3,xgboost,3.2.0,3.2.0
4,lightgbm,4.7.0,4.7.0


,feature_pack,derived_features
0,ratios,7
1,ratios_behavior,9


Pembanding selaras. Empat eksperimen siap dijalankan.


In [8]:
model_classes = {
    "xgb": XGBClassifier,
    "lgbm": LGBMClassifier,
}

refinement_results = {}
refinement_report_dirs = {}

for pack_name, features in feature_packs.items():
    for model_name, model_class in model_classes.items():
        experiment_id = f"{model_name}_aggregate_{pack_name}_v09"

        parameters = dict(
            reference_metadata["sources"][model_name]["parameters"]
        )

        pipeline = Pipeline([
            (
                "feature_engineering",
                CreditFeatureEngineer(features=features),
            ),
            (
                "preprocess",
                build_preprocessor(
                    numeric_scaling="none",
                    additional_numeric_columns=features,
                ).set_output(transform="pandas"),
            ),
            ("model", model_class(**parameters)),
        ])

        print(f"Menjalankan: {experiment_id}", flush=True)
        result = run_oof_experiment(pipeline, X_train, y_train)

        baseline = reference_oof[f"single_{model_name}_v08"]

        for column in ("source_row_index", "fold", "y_true"):
            np.testing.assert_array_equal(
                result.predictions[column].to_numpy(),
                baseline[column].to_numpy(),
            )

        report_dir = save_experiment(
            result=result,
            pipeline=pipeline,
            experiment_id=experiment_id,
            numeric_scaling="none",
        )

        refinement_results[experiment_id] = result
        refinement_report_dirs[experiment_id] = report_dir

        print(
            f"Selesai — cost/1000: "
            f"{result.summary['oof_cost_per_1000']:.3f}",
            flush=True,
        )

Menjalankan: xgb_aggregate_ratios_v09
Selesai — cost/1000: 533.125
Menjalankan: lgbm_aggregate_ratios_v09
Selesai — cost/1000: 538.083
Menjalankan: xgb_aggregate_ratios_behavior_v09
Selesai — cost/1000: 533.958
Menjalankan: lgbm_aggregate_ratios_behavior_v09
Selesai — cost/1000: 536.500


In [9]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)


def summarize_predictions(
    candidate: str,
    table: pd.DataFrame,
) -> dict[str, str | int | float]:
    """Ringkas prediksi OOF pada threshold terpilih."""
    target = table["y_true"].to_numpy()
    predicted = table["prediction"].to_numpy()
    probability = table["probability_default"].to_numpy()

    tn, fp, fn, tp = confusion_matrix(
        target, predicted, labels=[0, 1]
    ).ravel()

    return {
        "candidate": candidate,
        "cost_per_1000": (
            config.FALSE_NEGATIVE_COST * fn
            + config.FALSE_POSITIVE_COST * fp
        ) / len(table) * 1000,
        "roc_auc": roc_auc_score(target, probability),
        "average_precision": average_precision_score(target, probability),
        "f1": f1_score(target, predicted, zero_division=0),
        "fn": int(fn),
        "fp": int(fp),
        "recall_pct": tp / (tp + fn) * 100,
        "flagged_pct": predicted.mean() * 100,
    }


all_predictions = {
    **reference_oof,
    **{
        name: result.predictions
        for name, result in refinement_results.items()
    },
}

comparison = pd.DataFrame([
    summarize_predictions(name, table)
    for name, table in all_predictions.items()
]).sort_values("cost_per_1000")

# Kelompok yang menjadi fokus error analysis sebelumnya.
no_positive_repayment = (
    X_train.loc[:, list(config.REPAYMENT_STATUS_COLUMNS)]
    .le(0)
    .all(axis=1)
    .to_numpy()
)

subset_comparison = pd.DataFrame([
    summarize_predictions(
        name, table.loc[no_positive_repayment]
    )
    for name, table in all_predictions.items()
]).sort_values("cost_per_1000")

fold_rows = []

for name, result in refinement_results.items():
    model_name = name.split("_", maxsplit=1)[0]
    baseline = reference_oof[f"single_{model_name}_v08"]

    for fold in sorted(result.predictions["fold"].unique()):
        new_table = result.predictions.query("fold == @fold")
        old_table = baseline.query("fold == @fold")

        new_cost = summarize_predictions(name, new_table)["cost_per_1000"]
        old_cost = summarize_predictions("baseline", old_table)["cost_per_1000"]

        fold_rows.append({
            "candidate": name,
            "fold": int(fold),
            "delta_cost_per_1000": new_cost - old_cost,
        })

fold_deltas = pd.DataFrame(fold_rows).pivot(
    index="candidate",
    columns="fold",
    values="delta_cost_per_1000",
)

print("TABEL 1 — Semua development rows")
display(comparison.round(4))

print("TABEL 2 — Kelompok tanpa kode repayment positif")
display(subset_comparison.round(4))

print("TABEL 3 — Perubahan biaya terhadap baseline model yang sama")
print("Negatif berarti biaya lebih rendah; ini diagnostik development.")
display(fold_deltas.round(3))

# Simpan ringkasan bersama laporan masing-masing eksperimen.
for name, report_dir in refinement_report_dirs.items():
    comparison.to_csv(report_dir / "refinement_comparison.csv", index=False)
    subset_comparison.to_csv(
        report_dir / "refinement_subset_comparison.csv", index=False
    )
    fold_deltas.loc[[name]].to_csv(
        report_dir / "refinement_fold_deltas.csv"
    )

TABEL 1 — Semua development rows


,candidate,cost_per_1000,roc_auc,average_precision,f1,fn,fp,recall_pct,flagged_pct
3,xgb_aggregate_ratios_v09,533.1250,0.7887,0.5578,0.5012,1108,7255,79.1298,47.7333
5,xgb_aggregate_ratios_behavior_v09,533.9583,0.7888,0.5576,0.5005,1107,7280,79.1486,47.8417
2,blend_two_v08,534.7083,0.7878,0.5596,0.5142,1327,6198,75.0047,42.4167
0,single_xgb_v08,535.6250,0.7864,0.5562,0.5115,1299,6360,75.5321,43.2083
6,lgbm_aggregate_ratios_behavior_v09,536.5000,0.7890,0.5617,0.4972,1081,7471,79.6383,48.7458
4,lgbm_aggregate_ratios_v09,538.0833,0.7888,0.5613,0.5130,1348,6174,74.6092,42.2292
1,single_lgbm_v08,538.8750,0.7875,0.5606,0.5227,1489,5488,71.9533,38.7833


TABEL 2 — Kelompok tanpa kode repayment positif


,candidate,cost_per_1000,roc_auc,average_precision,f1,fn,fp,recall_pct,flagged_pct
3,xgb_aggregate_ratios_v09,522.1439,0.6668,0.2030,0.2793,986,3370,46.1202,26.5098
5,xgb_aggregate_ratios_behavior_v09,525.0377,0.6668,0.2060,0.2763,1002,3336,45.2459,26.1953
0,single_xgb_v08,526.0443,0.6592,0.2005,0.2644,1164,2542,36.3934,20.1812
1,single_lgbm_v08,526.6105,0.6616,0.2029,0.2485,1317,1786,28.0328,14.4628
2,blend_two_v08,526.6105,0.6622,0.2031,0.2607,1201,2366,34.3716,18.8412
4,lgbm_aggregate_ratios_v09,526.7363,0.6670,0.2051,0.2614,1192,2413,34.8634,19.1935
6,lgbm_aggregate_ratios_behavior_v09,527.1766,0.6678,0.2056,0.2759,981,3475,46.3934,27.2018


TABEL 3 — Perubahan biaya terhadap baseline model yang sama
Negatif berarti biaya lebih rendah; ini diagnostik development.


fold,1,2,3,4,5
candidate,,,,,
lgbm_aggregate_ratios_behavior_v09,0.625,-1.875,2.292,-14.792,1.875
lgbm_aggregate_ratios_v09,1.875,-9.375,-2.500,8.958,-2.917
xgb_aggregate_ratios_behavior_v09,2.292,7.500,-2.292,-3.125,-12.708
xgb_aggregate_ratios_v09,-7.917,2.708,2.917,-3.125,-7.083


### Hasil eksperimen fitur

XGBoost dengan fitur agregat dan rasio menghasilkan biaya OOF
terendah sebesar 533,125 per 1.000 observasi. Dibandingkan
XGBoost sebelumnya, biaya turun sekitar 0,47%, dengan peningkatan
kecil pada ROC-AUC dan Average Precision.

Dibandingkan kandidat blend sebelumnya, penurunan biaya hanya
sekitar 0,30%. Model mengurangi 219 false negative, tetapi
menambah 1.057 false positive. Proporsi prediksi positif meningkat
dari 42,42% menjadi 47,73%.

Pada kelompok tanpa kode repayment positif, ROC-AUC XGBoost
meningkat dari 0,6592 menjadi 0,6668. Ini menunjukkan peningkatan
ranking yang kecil. Kenaikan recall pada threshold terpilih juga
disertai peningkatan jumlah observasi yang ditandai berisiko.

Penambahan indikator pola pembayaran belum menghasilkan manfaat
yang konsisten. Perubahan biaya antar-fold juga bervariasi.
Karena itu, hasil ini belum membuktikan keunggulan generalisasi
kandidat baru.

Langkah berikutnya membandingkan model pada jumlah prediksi
positif yang sama dan menguji tiga bobot blend dari prediksi OOF
yang sudah tersedia. Test set tetap tidak digunakan.

In [10]:
from datetime import datetime, timezone

build_cost_curve = experiments.build_cost_curve

xgb_name = "xgb_aggregate_ratios_v09"
lgbm_name = "lgbm_aggregate_ratios_behavior_v09"

xgb_oof = all_predictions[xgb_name]
lgbm_oof = all_predictions[lgbm_name]

for column in ("source_row_index", "fold", "y_true"):
    np.testing.assert_array_equal(
        xgb_oof[column].to_numpy(),
        lgbm_oof[column].to_numpy(),
    )

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
blend_dir = config.EXPERIMENTS_DIR / "blend_refinement_v09" / run_id
blend_dir.mkdir(parents=True, exist_ok=False)

blend_predictions = {}
blend_thresholds = {}

for xgb_weight in (0.25, 0.50, 0.75):
    candidate = f"blend_refinement_xgb{int(xgb_weight * 100):02d}_v09"

    probabilities = (
        xgb_weight * xgb_oof["probability_default"].to_numpy()
        + (1 - xgb_weight)
        * lgbm_oof["probability_default"].to_numpy()
    )

    curve = build_cost_curve(
        xgb_oof["y_true"].to_numpy(),
        probabilities,
    )

    best = curve.sort_values(
        ["total_cost", "threshold"]
    ).iloc[0]
    threshold = float(best["threshold"])

    table = xgb_oof[
        ["source_row_index", "fold", "y_true"]
    ].copy()
    table["probability_default"] = probabilities
    table["prediction"] = (probabilities >= threshold).astype(int)

    blend_predictions[candidate] = table
    blend_thresholds[candidate] = threshold

    candidate_dir = blend_dir / candidate
    candidate_dir.mkdir()
    table.to_csv(candidate_dir / "oof_predictions.csv", index=False)
    curve.to_csv(candidate_dir / "threshold_curve.csv", index=False)

evaluated_predictions = {
    **all_predictions,
    **blend_predictions,
}

blend_comparison = pd.DataFrame([
    summarize_predictions(name, table)
    for name, table in evaluated_predictions.items()
]).sort_values("cost_per_1000")

print("TABEL 1 — Threshold minimum biaya masing-masing kandidat")
display(blend_comparison.round(4))

TABEL 1 — Threshold minimum biaya masing-masing kandidat


,candidate,cost_per_1000,roc_auc,average_precision,f1,fn,fp,recall_pct,flagged_pct
9,blend_refinement_xgb75_v09,532.1667,0.7895,0.5599,0.5080,1206,6742,77.2839,45.1875
8,blend_refinement_xgb50_v09,532.7917,0.7898,0.5614,0.5050,1167,6952,78.0185,46.2250
3,xgb_aggregate_ratios_v09,533.1250,0.7887,0.5578,0.5012,1108,7255,79.1298,47.7333
5,xgb_aggregate_ratios_behavior_v09,533.9583,0.7888,0.5576,0.5005,1107,7280,79.1486,47.8417
2,blend_two_v08,534.7083,0.7878,0.5596,0.5142,1327,6198,75.0047,42.4167
0,single_xgb_v08,535.6250,0.7864,0.5562,0.5115,1299,6360,75.5321,43.2083
7,blend_refinement_xgb25_v09,535.8333,0.7896,0.5620,0.5018,1151,7105,78.3198,46.9292
6,lgbm_aggregate_ratios_behavior_v09,536.5000,0.7890,0.5617,0.4972,1081,7471,79.6383,48.7458
4,lgbm_aggregate_ratios_v09,538.0833,0.7888,0.5613,0.5130,1348,6174,74.6092,42.2292
1,single_lgbm_v08,538.8750,0.7875,0.5606,0.5227,1489,5488,71.9533,38.7833


In [11]:
reference_flagged_count = int(
    reference_oof["blend_two_v08"]["prediction"].sum()
)


def apply_fixed_quota(
    table: pd.DataFrame,
    flagged_count: int,
) -> pd.DataFrame:
    """Tandai skor tertinggi dengan jumlah kasus yang ditetapkan."""
    result = table.copy()
    probabilities = result["probability_default"].to_numpy()
    source_rows = result["source_row_index"].to_numpy()

    # Jika skor sama, gunakan indeks sumber sebagai pemecah seri.
    order = np.lexsort((source_rows, -probabilities))
    predictions = np.zeros(len(result), dtype=int)
    predictions[order[:flagged_count]] = 1

    result["prediction"] = predictions
    return result


quota_comparison = pd.DataFrame([
    summarize_predictions(
        name,
        apply_fixed_quota(table, reference_flagged_count),
    )
    for name, table in evaluated_predictions.items()
]).sort_values("cost_per_1000")

blend_subset_comparison = pd.DataFrame([
    summarize_predictions(
        name,
        table.loc[no_positive_repayment],
    )
    for name, table in evaluated_predictions.items()
]).sort_values("cost_per_1000")

print(
    "TABEL 2 — Kuota sama:",
    reference_flagged_count,
    "prediksi positif per kandidat",
)
display(quota_comparison.round(4))

print("TABEL 3 — Kelompok tanpa kode repayment positif")
display(blend_subset_comparison.round(4))

blend_comparison.to_csv(blend_dir / "comparison.csv", index=False)
quota_comparison.to_csv(blend_dir / "quota_comparison.csv", index=False)
blend_subset_comparison.to_csv(
    blend_dir / "subset_comparison.csv", index=False
)

with (blend_dir / "metadata.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(
        {
            "evaluation_scope": "train_oof_development",
            "test_evaluated": False,
            "dataset_sha256": dataset_hash,
            "members": [xgb_name, lgbm_name],
            "xgb_weights": [0.25, 0.50, 0.75],
            "thresholds": blend_thresholds,
            "fixed_quota": reference_flagged_count,
            "member_reports": {
                name: str(
                    refinement_report_dirs[name].relative_to(
                        config.EXPERIMENTS_DIR
                    )
                )
                for name in (xgb_name, lgbm_name)
            },
        },
        file,
        indent=2,
        allow_nan=False,
    )

print("Ringkasan dan prediksi blend tersimpan.")

TABEL 2 — Kuota sama: 10180 prediksi positif per kandidat


,candidate,cost_per_1000,roc_auc,average_precision,f1,fn,fp,recall_pct,flagged_pct
2,blend_two_v08,534.7083,0.7878,0.5596,0.5142,1327,6198,75.0047,42.4167
0,single_xgb_v08,538.2083,0.7864,0.5562,0.5124,1341,6212,74.7410,42.4167
8,blend_refinement_xgb50_v09,538.2083,0.7898,0.5614,0.5124,1341,6212,74.7410,42.4167
3,xgb_aggregate_ratios_v09,538.4583,0.7887,0.5578,0.5122,1342,6213,74.7222,42.4167
4,lgbm_aggregate_ratios_v09,538.9583,0.7888,0.5613,0.5120,1344,6215,74.6845,42.4167
5,xgb_aggregate_ratios_behavior_v09,539.2083,0.7888,0.5576,0.5118,1345,6216,74.6657,42.4167
9,blend_refinement_xgb75_v09,539.2083,0.7895,0.5599,0.5118,1345,6216,74.6657,42.4167
7,blend_refinement_xgb25_v09,539.4583,0.7896,0.5620,0.5117,1346,6217,74.6468,42.4167
6,lgbm_aggregate_ratios_behavior_v09,540.9583,0.7890,0.5617,0.5109,1352,6223,74.5338,42.4167
1,single_lgbm_v08,541.2083,0.7875,0.5606,0.5108,1353,6224,74.5150,42.4167


TABEL 3 — Kelompok tanpa kode repayment positif


,candidate,cost_per_1000,roc_auc,average_precision,f1,fn,fp,recall_pct,flagged_pct
8,blend_refinement_xgb50_v09,521.5148,0.6690,0.2063,0.2763,1047,3055,42.7869,24.1444
3,xgb_aggregate_ratios_v09,522.1439,0.6668,0.2030,0.2793,986,3370,46.1202,26.5098
9,blend_refinement_xgb75_v09,522.6472,0.6684,0.2052,0.2735,1077,2923,41.1475,23.1253
5,xgb_aggregate_ratios_behavior_v09,525.0377,0.6668,0.2060,0.2763,1002,3336,45.2459,26.1953
7,blend_refinement_xgb25_v09,525.2894,0.6688,0.2065,0.2741,1036,3170,43.3880,24.9371
0,single_xgb_v08,526.0443,0.6592,0.2005,0.2644,1164,2542,36.3934,20.1812
2,blend_two_v08,526.6105,0.6622,0.2031,0.2607,1201,2366,34.3716,18.8412
1,single_lgbm_v08,526.6105,0.6616,0.2029,0.2485,1317,1786,28.0328,14.4628
4,lgbm_aggregate_ratios_v09,526.7363,0.6670,0.2051,0.2614,1192,2413,34.8634,19.1935
6,lgbm_aggregate_ratios_behavior_v09,527.1766,0.6678,0.2056,0.2759,981,3475,46.3934,27.2018


Ringkasan dan prediksi blend tersimpan.


### Kesimpulan weighted blend

Blend 75% XGBoost dan 25% LightGBM menghasilkan biaya OOF
terendah sebesar 532,167 per 1.000 observasi, sekitar 0,48%
lebih rendah daripada blend sebelumnya.

Perubahan tersebut mengurangi 121 false negative, tetapi
menambah 544 false positive. Proporsi prediksi positif naik
dari 42,42% menjadi 45,19%.

Pada kuota yang sama, blend sebelumnya menangkap 18 default
lebih banyak daripada blend baru. Dengan demikian, penurunan
biaya pada threshold terpilih tidak menunjukkan keunggulan
ranking yang konsisten di seluruh titik keputusan.

Kemampuan ranking pada kelompok tanpa kode repayment positif
meningkat sedikit, tetapi pemisahan kelas masih terbatas.

Blend baru dipertahankan sebagai kandidat biaya terbaik,
sedangkan blend sebelumnya tetap menjadi pembanding.
Hasil ini merupakan temuan development OOF dan belum menjadi
bukti keunggulan pada data baru.

In [12]:
importlib.reload(preprocess)
importlib.reload(experiments)

build_preprocessor = preprocess.build_preprocessor
run_oof_experiment = experiments.run_oof_experiment
save_experiment = experiments.save_experiment

representation_specs = {
    "xgb_repayment_numeric_v10": {
        "model": "xgb",
        "pack": "ratios",
        "baseline": "xgb_aggregate_ratios_v09",
    },
    "lgbm_repayment_numeric_v10": {
        "model": "lgbm",
        "pack": "ratios_behavior",
        "baseline": "lgbm_aggregate_ratios_behavior_v09",
    },
}

representation_results = {}
representation_report_dirs = {}

for experiment_id, spec in representation_specs.items():
    features = feature_packs[spec["pack"]]
    model_name = spec["model"]

    pipeline = Pipeline([
        (
            "feature_engineering",
            CreditFeatureEngineer(features=features),
        ),
        (
            "preprocess",
            build_preprocessor(
                numeric_scaling="none",
                additional_numeric_columns=features,
                repayment_encoding="numeric",
            ).set_output(transform="pandas"),
        ),
        (
            "model",
            model_classes[model_name](
                **dict(
                    reference_metadata["sources"][model_name]["parameters"]
                )
            ),
        ),
    ])

    print(f"Menjalankan: {experiment_id}", flush=True)
    result = run_oof_experiment(pipeline, X_train, y_train)

    baseline = all_predictions[spec["baseline"]]

    for column in ("source_row_index", "fold", "y_true"):
        np.testing.assert_array_equal(
            result.predictions[column].to_numpy(),
            baseline[column].to_numpy(),
        )

    report_dir = save_experiment(
        result=result,
        pipeline=pipeline,
        experiment_id=experiment_id,
        numeric_scaling="none",
    )

    representation_results[experiment_id] = result
    representation_report_dirs[experiment_id] = report_dir

    print(
        f"Selesai — cost/1000: "
        f"{result.summary['oof_cost_per_1000']:.3f}",
        flush=True,
    )

Menjalankan: xgb_repayment_numeric_v10
Selesai — cost/1000: 536.458
Menjalankan: lgbm_repayment_numeric_v10
Selesai — cost/1000: 536.375


In [13]:
representation_predictions = {
    name: result.predictions
    for name, result in representation_results.items()
}

final_development_predictions = {
    **evaluated_predictions,
    **representation_predictions,
}

representation_comparison = pd.DataFrame([
    summarize_predictions(name, table)
    for name, table in final_development_predictions.items()
]).sort_values("cost_per_1000")

representation_quota = pd.DataFrame([
    summarize_predictions(
        name,
        apply_fixed_quota(table, reference_flagged_count),
    )
    for name, table in final_development_predictions.items()
]).sort_values("cost_per_1000")

representation_fold_rows = []

for name, result in representation_results.items():
    baseline = all_predictions[
        representation_specs[name]["baseline"]
    ]

    for fold in sorted(result.predictions["fold"].unique()):
        new_table = result.predictions.query("fold == @fold")
        old_table = baseline.query("fold == @fold")

        new_cost = summarize_predictions(name, new_table)["cost_per_1000"]
        old_cost = summarize_predictions("baseline", old_table)["cost_per_1000"]

        representation_fold_rows.append({
            "candidate": name,
            "fold": int(fold),
            "delta_cost_per_1000": new_cost - old_cost,
        })

representation_fold_deltas = pd.DataFrame(
    representation_fold_rows
).pivot(
    index="candidate",
    columns="fold",
    values="delta_cost_per_1000",
)

print("TABEL 1 — Biaya pada threshold masing-masing kandidat")
display(representation_comparison.round(4))

print("TABEL 2 — Kuota prediksi positif yang sama")
display(representation_quota.round(4))

print("TABEL 3 — Delta biaya terhadap representasi OHE")
print("Negatif berarti lebih baik; diagnostik development.")
display(representation_fold_deltas.round(3))

for name, report_dir in representation_report_dirs.items():
    representation_comparison.to_csv(
        report_dir / "representation_comparison.csv", index=False
    )
    representation_quota.to_csv(
        report_dir / "representation_quota.csv", index=False
    )
    representation_fold_deltas.loc[[name]].to_csv(
        report_dir / "representation_fold_deltas.csv"
    )

TABEL 1 — Biaya pada threshold masing-masing kandidat


,candidate,cost_per_1000,roc_auc,average_precision,f1,fn,fp,recall_pct,flagged_pct
9,blend_refinement_xgb75_v09,532.1667,0.7895,0.5599,0.5080,1206,6742,77.2839,45.1875
8,blend_refinement_xgb50_v09,532.7917,0.7898,0.5614,0.5050,1167,6952,78.0185,46.2250
3,xgb_aggregate_ratios_v09,533.1250,0.7887,0.5578,0.5012,1108,7255,79.1298,47.7333
5,xgb_aggregate_ratios_behavior_v09,533.9583,0.7888,0.5576,0.5005,1107,7280,79.1486,47.8417
2,blend_two_v08,534.7083,0.7878,0.5596,0.5142,1327,6198,75.0047,42.4167
0,single_xgb_v08,535.6250,0.7864,0.5562,0.5115,1299,6360,75.5321,43.2083
7,blend_refinement_xgb25_v09,535.8333,0.7896,0.5620,0.5018,1151,7105,78.3198,46.9292
11,lgbm_repayment_numeric_v10,536.3750,0.7884,0.5634,0.5060,1225,6748,76.9260,45.1333
10,xgb_repayment_numeric_v10,536.4583,0.7871,0.5544,0.5085,1264,6555,76.1914,44.1667
6,lgbm_aggregate_ratios_behavior_v09,536.5000,0.7890,0.5617,0.4972,1081,7471,79.6383,48.7458


TABEL 2 — Kuota prediksi positif yang sama


,candidate,cost_per_1000,roc_auc,average_precision,f1,fn,fp,recall_pct,flagged_pct
2,blend_two_v08,534.7083,0.7878,0.5596,0.5142,1327,6198,75.0047,42.4167
0,single_xgb_v08,538.2083,0.7864,0.5562,0.5124,1341,6212,74.7410,42.4167
8,blend_refinement_xgb50_v09,538.2083,0.7898,0.5614,0.5124,1341,6212,74.7410,42.4167
3,xgb_aggregate_ratios_v09,538.4583,0.7887,0.5578,0.5122,1342,6213,74.7222,42.4167
4,lgbm_aggregate_ratios_v09,538.9583,0.7888,0.5613,0.5120,1344,6215,74.6845,42.4167
5,xgb_aggregate_ratios_behavior_v09,539.2083,0.7888,0.5576,0.5118,1345,6216,74.6657,42.4167
9,blend_refinement_xgb75_v09,539.2083,0.7895,0.5599,0.5118,1345,6216,74.6657,42.4167
7,blend_refinement_xgb25_v09,539.4583,0.7896,0.5620,0.5117,1346,6217,74.6468,42.4167
6,lgbm_aggregate_ratios_behavior_v09,540.9583,0.7890,0.5617,0.5109,1352,6223,74.5338,42.4167
1,single_lgbm_v08,541.2083,0.7875,0.5606,0.5108,1353,6224,74.5150,42.4167


TABEL 3 — Delta biaya terhadap representasi OHE
Negatif berarti lebih baik; diagnostik development.


fold,1,2,3,4,5
candidate,,,,,
lgbm_repayment_numeric_v10,-2.917,-10.208,3.125,14.167,-4.792
xgb_repayment_numeric_v10,8.958,8.750,-13.750,2.708,10.000


### Kesimpulan eksperimen representasi repayment

Representasi numerik tidak memberikan perbaikan biaya yang
meyakinkan dibandingkan OHE pada parameter yang diuji.

Pada XGBoost, biaya meningkat dari 533,125 menjadi 536,458
per 1.000 observasi dan memburuk pada empat dari lima fold.

Pada LightGBM, biaya turun dari 536,500 menjadi 536,375,
setara dengan tiga unit penalti pada seluruh development set.
Average Precision meningkat, tetapi hasil antar-fold bercampur
dan performa pada kuota yang sama memburuk.

OHE dipertahankan untuk kedua model. Kesimpulan ini berlaku
untuk konfigurasi yang diuji, bukan bukti bahwa representasi
numerik selalu lebih buruk.

Eksplorasi fitur, representasi repayment, dan bobot blend
dihentikan. Kandidat selanjutnya dinilai dari biaya, kalibrasi,
dan kompleksitas implementasi sebelum evaluasi test final.

### Pemeriksaan kalibrasi sebelum pemilihan model final

Pemeriksaan ini menggunakan prediksi OOF yang sudah tersedia
untuk XGBoost agregat + rasio dan blend 75/25.

Tujuannya menilai kualitas skor probabilitas dan menentukan
apakah kompleksitas blend memberikan manfaat tambahan.

Brier score dan log loss digunakan untuk menilai kualitas
prediksi probabilitas. Keduanya dipengaruhi kalibrasi serta
kemampuan pemisahan kelas, sehingga tidak ditafsirkan sebagai
ukuran kalibrasi murni.

Sebagai pembanding sederhana, probabilitas default konstan
untuk setiap validation fold dihitung dari prevalensi target
pada training fold lainnya.

Reliability table membandingkan rata-rata probabilitas dengan
frekuensi default dalam sepuluh interval probabilitas tetap.
Jumlah observasi ditampilkan untuk membantu menilai kestabilan
setiap interval.

Hasil ini merupakan diagnostik development. Tidak ada
kalibrator yang dilatih atau keputusan yang diubah pada tahap ini.
Test set tetap dikarantina.

In [14]:
from sklearn.metrics import brier_score_loss, log_loss

calibration_candidates = (
    "xgb_aggregate_ratios_v09",
    "blend_refinement_xgb75_v09",
)

calibration_tables = {
    name: final_development_predictions[name].copy()
    for name in calibration_candidates
}

reference = calibration_tables[calibration_candidates[0]]
target = reference["y_true"].to_numpy(dtype=int)
fold_ids = reference["fold"].to_numpy()

for table in calibration_tables.values():
    for column in ("source_row_index", "fold", "y_true"):
        np.testing.assert_array_equal(
            table[column].to_numpy(),
            reference[column].to_numpy(),
        )

# Pembanding probabilitas konstan yang tidak memakai label
# validation fold untuk menentukan probabilitasnya.
prior_probabilities = np.empty(len(reference), dtype=float)

for fold in np.unique(fold_ids):
    validation_mask = fold_ids == fold
    prior_probabilities[validation_mask] = target[
        ~validation_mask
    ].mean()


def probability_metrics(
    candidate: str,
    probabilities: np.ndarray,
) -> dict[str, str | float]:
    """Ringkas kualitas probabilitas terhadap target OOF."""
    if not np.isfinite(probabilities).all():
        raise ValueError("Probabilitas mengandung nilai non-finite.")

    if ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Probabilitas harus berada pada rentang 0–1.")

    return {
        "candidate": candidate,
        "brier_score": brier_score_loss(target, probabilities),
        "log_loss": log_loss(target, probabilities, labels=[0, 1]),
        "mean_probability_pct": probabilities.mean() * 100,
        "observed_default_pct": target.mean() * 100,
        "mean_gap_pp": (probabilities.mean() - target.mean()) * 100,
    }


probability_rows = [
    probability_metrics("fold_training_prevalence", prior_probabilities)
]
reliability_parts = []

for name, table in calibration_tables.items():
    probabilities = table["probability_default"].to_numpy()
    probability_rows.append(probability_metrics(name, probabilities))

    table["probability_bin"] = pd.cut(
        probabilities,
        bins=np.linspace(0, 1, 11),
        include_lowest=True,
        right=True,
        precision=3,
    )

    reliability = (
        table.groupby("probability_bin", observed=True)
        .agg(
            rows=("y_true", "size"),
            defaults=("y_true", "sum"),
            mean_probability=("probability_default", "mean"),
            observed_default_rate=("y_true", "mean"),
        )
        .reset_index()
    )

    reliability.insert(0, "candidate", name)
    reliability["probability_bin"] = (
        reliability["probability_bin"].astype(str)
    )
    reliability["mean_probability_pct"] = (
        reliability.pop("mean_probability") * 100
    )
    reliability["observed_default_pct"] = (
        reliability.pop("observed_default_rate") * 100
    )
    reliability["gap_pp"] = (
        reliability["mean_probability_pct"]
        - reliability["observed_default_pct"]
    )

    reliability_parts.append(reliability)

probability_summary = pd.DataFrame(probability_rows)
reliability_table = pd.concat(reliability_parts, ignore_index=True)

print("TABEL 1 — Kualitas probabilitas; Brier dan log loss lebih kecil lebih baik")
display(probability_summary.round(5))

print("TABEL 2 — Reliability; gap positif berarti risiko diprediksi lebih tinggi")
display(reliability_table.round(3))

TABEL 1 — Kualitas probabilitas; Brier dan log loss lebih kecil lebih baik


,candidate,brier_score,log_loss,mean_probability_pct,observed_default_pct,mean_gap_pp
0,fold_training_prevalence,0.17228,0.52843,22.12083,22.12083,0.00000
1,xgb_aggregate_ratios_v09,0.13328,0.42475,22.11696,22.12083,-0.00387
2,blend_refinement_xgb75_v09,0.13304,0.42393,22.10503,22.12083,-0.01581


TABEL 2 — Reliability; gap positif berarti risiko diprediksi lebih tinggi


,candidate,probability_bin,rows,defaults,mean_probability_pct,observed_default_pct,gap_pp
0,xgb_aggregate_ratios_v09,"(-0.001, 0.1]",7276,461,6.689,6.336,0.353
1,xgb_aggregate_ratios_v09,"(0.1, 0.2]",8526,1243,14.344,14.579,-0.235
2,xgb_aggregate_ratios_v09,"(0.2, 0.3]",2936,725,24.121,24.693,-0.573
3,xgb_aggregate_ratios_v09,"(0.3, 0.4]",1533,534,34.633,34.834,-0.201
4,xgb_aggregate_ratios_v09,"(0.4, 0.5]",857,394,44.637,45.974,-1.338
5,xgb_aggregate_ratios_v09,"(0.5, 0.6]",711,415,55.005,58.368,-3.363
6,xgb_aggregate_ratios_v09,"(0.6, 0.7]",818,532,65.176,65.037,0.140
7,xgb_aggregate_ratios_v09,"(0.7, 0.8]",842,623,74.994,73.990,1.004
8,xgb_aggregate_ratios_v09,"(0.8, 0.9]",473,356,83.586,75.264,8.321
9,xgb_aggregate_ratios_v09,"(0.9, 1.0]",28,26,91.956,92.857,-0.901


In [15]:
calibration_run_id = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)
calibration_dir = (
    config.EXPERIMENTS_DIR
    / "calibration_audit_v11"
    / calibration_run_id
)
calibration_dir.mkdir(parents=True, exist_ok=False)

probability_summary.to_csv(
    calibration_dir / "probability_metrics.csv", index=False
)
reliability_table.to_csv(
    calibration_dir / "reliability_table.csv", index=False
)

with (calibration_dir / "metadata.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(
        {
            "evaluation_scope": "development_oof_diagnostic",
            "dataset_sha256": dataset_hash,
            "candidates": list(calibration_candidates),
            "bin_edges": np.linspace(0, 1, 11).tolist(),
            "baseline": "prevalence_from_other_training_folds",
            "calibrator_fitted": False,
            "test_evaluated": False,
            "versions": {
                package: version(package)
                for package in ("numpy", "pandas", "scikit-learn")
            },
        },
        file,
        indent=2,
        allow_nan=False,
    )

print("Laporan kalibrasi tersimpan.")

Laporan kalibrasi tersimpan.


# Model final dan evaluasi holdout

Kandidat yang dipilih adalah weighted soft voting:
75% XGBoost dan 25% LightGBM.

XGBoost menggunakan fitur asli, agregat, dan rasio.
LightGBM menggunakan fitur asli, agregat, rasio, dan indikator
pola pembayaran. Repayment tetap menggunakan One-Hot Encoding.

Pemilihan dilakukan berdasarkan development OOF. Blend memberikan
perbaikan kecil pada biaya dan kualitas probabilitas dibandingkan
XGBoost tunggal, dengan konsekuensi kompleksitas dua model.

Parameter, fitur, bobot, dan threshold OOF dikunci sebelum
evaluasi test. Tidak ada tuning tambahan atau kalibrasi tambahan.

Kedua pipeline dilatih pada seluruh development set.
Artefak diverifikasi melalui penyimpanan dan pemuatan kembali
sebelum test set dievaluasi.

In [16]:
import hashlib
import json
from datetime import datetime, timezone
from importlib.metadata import version
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from src import config
from src.data_loader import load_and_split
from src.feature_engineering import CreditFeatureEngineer
from src.preprocess import build_preprocessor, get_feature_mapping

FINAL_CANDIDATE = "blend_refinement_xgb75_v09"

# Gunakan laporan blend terbaru yang sudah dibuat.
blend_runs = sorted(
    path
    for path in (
        config.EXPERIMENTS_DIR / "blend_refinement_v09"
    ).glob("*")
    if path.is_dir() and (path / "metadata.json").is_file()
)
if not blend_runs:
    raise FileNotFoundError("Laporan blend v09 tidak ditemukan.")

blend_report = blend_runs[-1]

with (blend_report / "metadata.json").open(
    encoding="utf-8"
) as file:
    blend_metadata = json.load(file)

dataset_hash = hashlib.sha256(
    config.DATA_PATH.read_bytes()
).hexdigest()
assert dataset_hash == blend_metadata["dataset_sha256"]

locked_threshold = float(
    blend_metadata["thresholds"][FINAL_CANDIDATE]
)

member_specs = {}
for member_name in blend_metadata["members"]:
    report_dir = (
        config.EXPERIMENTS_DIR
        / blend_metadata["member_reports"][member_name]
    )

    with (report_dir / "metadata.json").open(
        encoding="utf-8"
    ) as file:
        member_metadata = json.load(file)

    assert member_metadata["dataset_sha256"] == dataset_hash
    assert member_metadata["categorical_encoding"] == "one_hot"
    assert member_metadata["split_random_state"] == config.RANDOM_STATE
    assert member_metadata["test_size"] == config.TEST_SIZE
    assert member_metadata["false_negative_cost"] == config.FALSE_NEGATIVE_COST
    assert member_metadata["false_positive_cost"] == config.FALSE_POSITIVE_COST

    for package, expected_version in member_metadata["versions"].items():
        if version(package) != expected_version:
            raise RuntimeError(
                f"Versi {package} berubah: "
                f"{expected_version} → {version(package)}."
            )

    member_specs[member_name] = member_metadata

assert blend_metadata["members"] == [
    "xgb_aggregate_ratios_v09",
    "lgbm_aggregate_ratios_behavior_v09",
]

display(pd.DataFrame([
    {
        "candidate": FINAL_CANDIDATE,
        "xgb_weight": 0.75,
        "lgbm_weight": 0.25,
        "locked_threshold": locked_threshold,
        "test_evaluated": False,
    }
]))

,candidate,xgb_weight,lgbm_weight,locked_threshold,test_evaluated
0,blend_refinement_xgb75_v09,0.75,0.25,0.158931,False


In [18]:
def restore_parameter_values(value: object) -> object:
    """Restore non-finite floats encoded in experiment metadata."""
    if isinstance(value, dict):
        if set(value) == {"__non_finite_float__"}:
            return float(value["__non_finite_float__"])

        return {
            key: restore_parameter_values(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [restore_parameter_values(item) for item in value]

    return value


restored_model_parameters = {
    name: restore_parameter_values(specification["model_parameters"])
    for name, specification in member_specs.items()
}

xgb_parameters = restored_model_parameters[
    "xgb_aggregate_ratios_v09"
]
assert isinstance(xgb_parameters, dict)
assert np.isnan(xgb_parameters["missing"])

print("Parameter missing XGBoost berhasil dipulihkan menjadi np.nan.")

Parameter missing XGBoost berhasil dipulihkan menjadi np.nan.


In [ ]:
split = load_and_split()

estimators = []

for short_name, member_name, model_class in (
    ("xgb", "xgb_aggregate_ratios_v09", XGBClassifier),
    ("lgbm", "lgbm_aggregate_ratios_behavior_v09", LGBMClassifier),
):
    specification = member_specs[member_name]
    features = tuple(
        specification["feature_engineering_parameters"]["features"]
    )

    pipeline = Pipeline([
        (
            "feature_engineering",
            CreditFeatureEngineer(features=features),
        ),
        (
            "preprocess",
            build_preprocessor(
                numeric_scaling="none",
                additional_numeric_columns=features,
                repayment_encoding="one_hot",
            ).set_output(transform="pandas"),
        ),
        (
            "model",
            model_class(**restored_model_parameters[member_name]),
        ),
    ])
    estimators.append((short_name, pipeline))

final_model = VotingClassifier(
    estimators=estimators,
    voting="soft",
    weights=[0.75, 0.25],
    n_jobs=1,
)

final_dir = config.MODELS_DIR / FINAL_CANDIDATE

if final_dir.exists():
    raise FileExistsError(
        f"Folder sudah ada: {final_dir}. "
        "Artefak final tidak ditimpa otomatis."
    )

print("Fit dua pipeline pada seluruh development set...", flush=True)
started = perf_counter()

final_model.fit(split.X_train, split.y_train)

assert final_model.classes_.tolist() == [0, 1]
fit_seconds = perf_counter() - started

# Probe dari development; test belum diprediksi.
probe = split.X_train.sample(
    n=64, random_state=config.RANDOM_STATE
)

before_save = final_model.predict_proba(probe)

# Pastikan VotingClassifier menghasilkan formula blend yang dikunci.
manual_default_probability = (
    0.75
    * final_model.named_estimators_["xgb"].predict_proba(probe)[:, 1]
    + 0.25
    * final_model.named_estimators_["lgbm"].predict_proba(probe)[:, 1]
)
np.testing.assert_allclose(
    before_save[:, 1],
    manual_default_probability,
    rtol=0,
    atol=1e-12,
)

final_dir.mkdir(parents=True, exist_ok=False)
model_path = final_dir / "model.joblib"

joblib.dump(
    {
        "model": final_model,
        "threshold": locked_threshold,
        "positive_class": 1,
        "candidate": FINAL_CANDIDATE,
    },
    model_path,
    compress=3,
)

loaded_bundle = joblib.load(model_path)
after_save = loaded_bundle["model"].predict_proba(probe)

np.testing.assert_allclose(
    before_save, after_save, rtol=0, atol=1e-12
)

for name, pipeline in final_model.named_estimators_.items():
    get_feature_mapping(
        pipeline.named_steps["preprocess"]
    ).to_csv(
        final_dir / f"feature_mapping_{name}.csv",
        index=False,
    )

final_metadata = {
    "candidate": FINAL_CANDIDATE,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "dataset_sha256": dataset_hash,
    "threshold": locked_threshold,
    "threshold_source": "development_oof_minimum_cost",
    "weights": {"xgb": 0.75, "lgbm": 0.25},
    "positive_class": 1,
    "train_rows": len(split.X_train),
    "test_evaluated": False,
    "fit_seconds": fit_seconds,
    "split_random_state": config.RANDOM_STATE,
    "test_size": config.TEST_SIZE,
    "false_negative_cost": config.FALSE_NEGATIVE_COST,
    "false_positive_cost": config.FALSE_POSITIVE_COST,
    "member_specifications": member_specs,
    "source_sha256": {
        filename: hashlib.sha256(
            (config.PROJECT_ROOT / "src" / filename).read_bytes()
        ).hexdigest()
        for filename in (
            "config.py",
            "feature_engineering.py",
            "preprocess.py",
        )
    },
}

with (final_dir / "metadata.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(final_metadata, file, indent=2, allow_nan=False)

print(f"Fit selesai: {fit_seconds:.1f} detik")
print("Formula blend dan hasil setelah load: cocok.")
print(f"Threshold terkunci: {locked_threshold:.10f}")
print(f"Artefak: {final_dir}")
print("Test set belum dievaluasi.")

Fit dua pipeline pada seluruh development set...


XGBoostError: [16:28:16] C:\actions-runner\_work\xgboost\xgboost\src\c_api\../common/json_utils.h:48: Invalid type for: `missing`, expecting one of the: {`Number`, `Integer`}, got: `Object`